# Line of maximum reconnection rate

Minimal extract of `Rate_vs_Tracers_all_events.ipynb`: compute the 2D reconnection-rate map on the magnetopause and the line of maximum rate starting from the max point.

The max line is computed with `max_rate_line.make_max_qty_from_max_point` (in this folder) instead of `shear_maps.make_max_qty_from_max_point`.
`shear_maps` relies on a bug of `skimage.feature.hessian_matrix` in scikit-image <= 0.19 (`order='rc'` returned the axes in reversed order). That bug was fixed in scikit-image 0.20, so with recent versions the y/z components of the Hessian eigenvectors are swapped and the line goes in the wrong direction.
`max_rate_line` computes the Hessian explicitly (same computation as scikit-image 0.19), so it gives the old results whatever scikit-image version is installed.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import constants as cst
from matplotlib.colors import Normalize

sys.path.append('/DATA/michotte/xline/')

from spok.models import planetary as smp
from spok.coordinates import coordinates as scc
from spok import utils as su
import shear_maps as smap
import maps_mp_data as mmp

import max_rate_line as mrl

msh = smp.Magnetosheath(magnetopause='mp_shue1998', bow_shock='bs_jelinek2012')

In [ ]:
def fig_nxm_plots(n=2, m=2, figsize=None, msh=msh):
    mp, bs = msh.boundaries(np.pi/2, np.linspace(0, 2*np.pi, 50))
    if figsize is None:
        fig, axs = plt.subplots(m, n, figsize=(n*7., m*6+0.9))
    else:
        fig, axs = plt.subplots(m, n, figsize=figsize)
    axs = np.asarray(su.listify(axs))
    for ax in axs.ravel():
        ax.plot(mp[1], mp[2], ls='-.', color='k')
        ax.plot(bs[1], bs[2], ls='--', color='k')
        ax.set_aspect('equal')
        ax.axhline(0, ls='--', c='k', alpha=0.75)
        ax.axvline(0, ls='--', c='k', alpha=0.75)
        ax.set_xlim(-16, 16)
        ax.set_ylim(-16, 16)
        ax.tick_params(axis='both', which='major', labelsize=13)
        ax.set_xlabel(r"$Y_{PGSM}$", fontsize=18)
    axs.ravel()[0].set_ylabel(r" $Z_{PGSM}$", fontsize=18)
    fig.tight_layout()
    return fig, axs


def swi_to_negative_bximf(xmp, ymp, zmp, bx, by, bz):
    ymp, zmp = -ymp, -zmp
    bx = -bx
    return xmp, ymp, zmp, bx, by, bz


def make_bisec_reconnection_rate(coa, cla, ps_deg, bimf_norm=5, np_sw=5):
    bxmsp, bymsp, bzmsp = b_msp_tilt[str(ps_deg)]
    npmsp = np_msp_tilt[str(ps_deg)]*1e6*cst.m_p

    npmsh0 = np_msh_coa[str(abs(coa))]*1e6*cst.m_p*np_sw
    bxmsh0, bymsh0, bzmsh0 = grid_bmsh_coa[str(abs(coa))]
    x0, y0, z0 = Xmp.copy(), Ymp.copy(), Zmp.copy()
    if np.sign(coa) == -1:
        x0, y0, z0, bxmsh0, bymsh0, bzmsh0 = swi_to_negative_bximf(x0, y0, z0, bxmsh0, bymsh0, bzmsh0)
        npmsh0 = smap.interpolate_on_regular_grid(-y0, -y0, [npmsh0], y0, y0)

    new_clock = np.radians(cla)
    old_clock = np.radians(90)
    xr, yr, zr, bxmsh1, bymsh1, bzmsh1 = smap.rotates_clock_angle(x0, y0, z0, bxmsh0, bymsh0, bzmsh0, new_clock, old_clock)
    bxmsh, bymsh, bzmsh = smap.interpolate_on_regular_grid(yr, zr, [bxmsh1, bymsh1, bzmsh1], yy, zz)
    npmsh = smap.interpolate_on_regular_grid(yr, zr, [npmsh0], yy, zz)[0]
    bxmsh, bymsh, bzmsh, npmsh = smap.make_gaussian_filter([bxmsh, bymsh, bzmsh, npmsh], (20, 20))

    bxmsh, bymsh, bzmsh = smap.remove_normal_to_shue98(xx, yy, zz, bxmsh, bymsh, bzmsh)

    theta = smap.shear_angle(bxmsp, bymsp, bzmsp, bxmsh, bymsh, bzmsh)/2
    rr = mmp.RR_theta(theta, npmsp, npmsh, [bxmsp, bymsp, bzmsp], [bxmsh, bymsh, bzmsh], bimf_norm=bimf_norm)

    th_bmsp = np.degrees(np.arctan2(bymsp, bzmsp))
    th_bmsh = np.degrees(np.arctan2(bymsh, bzmsh))
    sign = th_bmsh-th_bmsp
    sign[sign > 180] -= 360
    phi_th = th_bmsp + np.sign(sign)*theta
    vec_field = np.sin(np.radians(phi_th)), np.cos(np.radians(phi_th))
    return rr, theta, vec_field

In [ ]:
# Magnetopause and regular (Y, Z) grid
N = 300
th = np.linspace(0, 0.95*np.pi, N)
ph = np.linspace(0, 2*np.pi, 2*N)
theta, phi = np.meshgrid(th, ph)
Xmp, Ymp, Zmp = msh.magnetopause(theta, phi)

yy, zz = smap.make_regular_grid(xlim=(-22, 22), ylim=(-22, 22), nb_pts=401)
xx = smap.interpolate_on_regular_grid(Ymp, Zmp, [Xmp], yy, zz)[0]

In [ ]:
grid_bmsh_coa = pd.read_pickle('/DATA/michotte/xline/dico_mp_b_msh.pkl')
np_msh_coa = pd.read_pickle('/DATA/michotte/xline/dico_mp_np_msh.pkl')
b_msp_tilt = pd.read_pickle('/DATA/michotte/xline/dico_mp_b_msp.pkl')
np_msp_tilt = pd.read_pickle('/DATA/michotte/xline/dico_mp_np_msp.pkl')

In [ ]:
# Max rate line
coa, tilt = 75, 0
clas = [0, 45, 90, 135, 180]
save = False  # set to True to save the lines as before

fig, axs = fig_nxm_plots(len(clas), 1, figsize=(6*len(clas), 5.5))
lines = {}
for ax, cla in zip(axs, clas):
    rate, theta, vec = make_bisec_reconnection_rate(coa, cla, tilt, bimf_norm=5, np_sw=5)
    line = mrl.make_max_qty_from_max_point(yy, zz, rate)  # if only one max point
    lines[cla] = line

    c = ax.pcolormesh(yy, zz, rate, cmap='jet', norm=Normalize(0, rate.max()))
    ax.plot(line[0], line[1], c='k', lw=2)
    ax.set_title(f'clock = {cla}°, cone = {coa}°, tilt = {tilt}°')
    if save:
        pd.to_pickle({'CLA': cla, 'COA': coa, 'tilt': tilt, 'y': line[0], 'z': line[1], 'method': 'max_rate'},
                     f'/home/ghisalberti/komar_separators/max_rate_cla_{cla}_coa_{coa}_tilt_{tilt}.pkl')
fig.colorbar(c, ax=axs[-1], label='Rate')